In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import plotly.express as px
import plotly.figure_factory as ff

In [3]:
# Load data
data = np.load('clip_embeddings_uwisc_east_2025-01-09.npz', allow_pickle=True)
X = data['embeddings']
idents = data['idents']
timestamps = data['timestamps']
y = data['is_contrail'].astype(int)

class_names = ['No Contrail', 'Contrail']

print(f"Samples: {len(X)}, Features: {X.shape[1]}")
print(f"Distribution: No Contrail={np.sum(y==0)}, Contrail={np.sum(y==1)}")

Samples: 2131, Features: 512
Distribution: No Contrail=1800, Contrail=331


In [4]:
# Split by flight ident so same flight doesn't appear in both train and test
unique_idents = np.unique(idents)
train_idents, test_idents = train_test_split(unique_idents, test_size=0.2, random_state=42)

train_mask = np.isin(idents, train_idents)
test_mask = np.isin(idents, test_idents)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {len(X_train)} samples ({len(train_idents)} flights)")
print(f"Test:  {len(X_test)} samples ({len(test_idents)} flights)")

Train: 1729 samples (124 flights)
Test:  402 samples (32 flights)


In [5]:
# Create dataloaders
train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
# Binary linear classifier
model = nn.Linear(512, 2)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

Parameters: 1026


In [7]:
# Training loop
num_epochs = 50
train_losses = []
test_losses = []
test_accs = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    train_losses.append(epoch_loss / len(X_train))

    model.eval()
    correct = 0
    total_loss = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            out = model(xb)
            total_loss += criterion(out, yb).item() * len(xb)
            correct += (out.argmax(1) == yb).sum().item()
    test_losses.append(total_loss / len(X_test))
    test_accs.append(correct / len(X_test))

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_losses[-1]:.4f} | Test Loss: {test_losses[-1]:.4f} | Test Acc: {test_accs[-1]:.3f}")

Epoch  10 | Train Loss: 0.2908 | Test Loss: 0.3482 | Test Acc: 0.799
Epoch  20 | Train Loss: 0.1998 | Test Loss: 0.2465 | Test Acc: 0.888
Epoch  30 | Train Loss: 0.1474 | Test Loss: 0.1885 | Test Acc: 0.923
Epoch  40 | Train Loss: 0.1143 | Test Loss: 0.1498 | Test Acc: 0.953
Epoch  50 | Train Loss: 0.0921 | Test Loss: 0.1251 | Test Acc: 0.958


In [8]:
# Plot training curves
fig = px.line(
    pd.DataFrame({'train': train_losses, 'test': test_losses, 'epoch': range(1, num_epochs+1)}),
    x='epoch', y=['train', 'test'],
    title='Loss Curves',
    labels={'value': 'Loss', 'variable': 'Split'}
)
fig.show()

fig2 = px.line(x=range(1, num_epochs+1), y=test_accs, title='Test Accuracy', labels={'x': 'Epoch', 'y': 'Accuracy'})
fig2.show()

In [9]:
# Evaluation
model.eval()
with torch.no_grad():
    all_preds = model(torch.FloatTensor(X_test)).argmax(1).numpy()

print(classification_report(y_test, all_preds, target_names=class_names))

              precision    recall  f1-score   support

 No Contrail       0.95      1.00      0.97       321
    Contrail       1.00      0.79      0.88        81

    accuracy                           0.96       402
   macro avg       0.97      0.90      0.93       402
weighted avg       0.96      0.96      0.96       402



In [10]:
# Confusion matrix
cm = confusion_matrix(y_test, all_preds)
fig = ff.create_annotated_heatmap(
    cm, x=class_names, y=class_names,
    colorscale='Blues', showscale=True
)
fig.update_layout(title='Confusion Matrix', xaxis_title='Predicted', yaxis_title='Actual')
fig.show()

In [14]:
# Accuracy against human labels (aggregated per flight)
# Human labels are per-ident: if a plane makes a contrail at any point, it's labeled as contrail
labels_df = pd.read_csv('labels/UWisc Aoss Contrail Labels - contrail_labels_2025-01-09_east.csv')
ident_to_label = dict(zip(labels_df['ident'], labels_df['label']))
human_labels = np.array([ident_to_label.get(ident, None) for ident in idents])

has_label = human_labels != None
X_labeled = X[has_label]
idents_labeled = idents[has_label]
human_labels_filtered = human_labels[has_label]

# Run classifier on all labeled samples
model.eval()
with torch.no_grad():
    preds = model(torch.FloatTensor(X_labeled)).argmax(1).numpy()

contrail_labels = {'Persistent', 'Dissipate < 10', 'Dissipate > 10'}

# Aggregate per flight: if classifier predicts contrail on ANY frame -> flight = contrail
flight_df = pd.DataFrame({
    'ident': idents_labeled,
    'human_label': human_labels_filtered,
    'pred': preds,
})

flight_agg = flight_df.groupby('ident').agg(
    human_label=('human_label', 'first'),
    pred_any_contrail=('pred', 'max'),
    pred_contrail_frac=('pred', 'mean'),
    n_frames=('pred', 'count'),
).reset_index()

flight_agg['human_binary'] = flight_agg['human_label'].isin(contrail_labels).astype(int)

# Overall flight-level accuracy
acc = (flight_agg['pred_any_contrail'] == flight_agg['human_binary']).mean()
print(f"Flight-level accuracy: {acc:.1%} ({len(flight_agg)} flights)\n")

print(classification_report(
    flight_agg['human_binary'], flight_agg['pred_any_contrail'],
    target_names=['No Contrail (Clear/Blocked)', 'Contrail (Persistent/Dissipate)']
))

# Confusion matrix
cm = confusion_matrix(flight_agg['human_binary'], flight_agg['pred_any_contrail'])
fig = ff.create_annotated_heatmap(
    cm,
    x=['Pred: No Contrail', 'Pred: Contrail'],
    y=['Human: No Contrail', 'Human: Contrail'],
    colorscale='Blues', showscale=True
)
fig.update_layout(title='Flight-Level: Classifier vs Human Labels', width=600, height=500)
fig.show()

# Per human label breakdown
print("\nPer human label (flight-level):")
for label in ['Persistent', 'Dissipate > 10', 'Dissipate < 10', 'Clear', 'Blocked']:
    mask = flight_agg['human_label'] == label
    expected = 1 if label in contrail_labels else 0
    if mask.sum() > 0:
        acc_label = (flight_agg.loc[mask, 'pred_any_contrail'] == expected).mean()
        avg_frac = flight_agg.loc[mask, 'pred_contrail_frac'].mean()
        print(f"  {label:>16s}: {acc_label:.1%} correct ({mask.sum()} flights, avg {avg_frac:.0%} frames predicted contrail)")

Flight-level accuracy: 86.5% (156 flights)

                                 precision    recall  f1-score   support

    No Contrail (Clear/Blocked)       0.85      0.95      0.90        96
Contrail (Persistent/Dissipate)       0.90      0.73      0.81        60

                       accuracy                           0.87       156
                      macro avg       0.87      0.84      0.85       156
                   weighted avg       0.87      0.87      0.86       156




Per human label (flight-level):
        Persistent: 63.6% correct (22 flights, avg 23% frames predicted contrail)
    Dissipate > 10: 75.0% correct (4 flights, avg 39% frames predicted contrail)
    Dissipate < 10: 79.4% correct (34 flights, avg 32% frames predicted contrail)
             Clear: 92.1% correct (63 flights, avg 2% frames predicted contrail)
           Blocked: 100.0% correct (33 flights, avg 0% frames predicted contrail)


In [ ]:
# Save the trained classifier for use in pipelines
torch.save(model.state_dict(), 'clip_contrail_classifier.pt')
print("Saved classifier to clip_contrail_classifier.pt")